# VL-JEPA Proof of Concept

Toy implementation of VL-JEPA architecture from the paper.

## Architecture Components

- **X-Encoder**: V-JEPA2 ViT-L (304M params, frozen)
- **Predictor**: Last 8 layers of Llama-3.2-1B (490M params, trainable)
- **Y-Encoder**: EmbeddingGemma-300M (trainable, 0.05× LR multiplier)
- **Loss**: InfoNCE (bidirectional contrastive)

## Key Architectural Decisions

1. Joint training of Predictor + Y-Encoder (not post-hoc alignment)
2. Non-autoregressive prediction (bidirectional attention)
3. Y-Encoder uses lower learning rate for stability
4. InfoNCE loss in embedding space (not token space)

In [1]:
%pip install torch torchvision torchaudio transformers

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 12.0/12.0 MB 93.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   --------------------------------------- 566.1/566.1 kB 20.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 80.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\cddal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Setup

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModel,
    AutoTokenizer,
    LlamaForCausalLM,
)
from typing import Optional, Tuple
import math

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Memory settings for 8GB VRAM target
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Using device: cpu


## Memory Optimization Strategies

### LoRA (Low-Rank Adaptation)

LoRA is ideal for this architecture:

**Why LoRA Works Well Here:**
- Predictor (Llama layers) naturally supports LoRA adapters
- Y-Encoder can use LoRA on attention layers
- X-Encoder already frozen (no adaptation needed)
- Reduces trainable params by 10-100x
- Maintains full model quality

**Implementation:**
```python
from peft import LoraConfig, get_peft_model, TaskType

# LoRA config for Predictor
lora_config = LoraConfig(
    r=16,  # rank (paper suggests 8-32)
    lora_alpha=32,  # scaling factor
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # attention layers
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION
)

# Apply to predictor
predictor = get_peft_model(predictor, lora_config)
predictor.print_trainable_parameters()  # Shows dramatic reduction
```

**Expected savings:**
- Predictor: 490M → ~10M trainable params
- Y-Encoder: 300M → ~5M trainable params
- Total training memory: ~5GB → ~2GB


### QLoRA (Quantized LoRA)

Combine quantization + LoRA for maximum efficiency:

**Benefits:**
- Load base models in 4-bit quantization
- Train LoRA adapters in higher precision
- Fits comfortably in 8GB VRAM
- Minimal quality degradation

**Implementation:**
```python
from transformers import BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',  # normalized float 4
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # nested quantization
)

# Load Llama with quantization
llama_model = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    quantization_config=bnb_config,
    device_map='auto',
)

# Then apply LoRA on top
predictor = get_peft_model(llama_model, lora_config)
```

**Memory with QLoRA:**
- Predictor: 980MB → 245MB (4-bit) + 10MB (LoRA) = 255MB
- Y-Encoder: 600MB → 150MB (4-bit) + 5MB (LoRA) = 155MB
- X-Encoder: 608MB (frozen, can also quantize)
- Total: ~1GB models + ~3GB activations = **4GB VRAM**


### UnslothAI Integration

UnslothAI provides optimized training for Llama models:

**Key Features:**
- 2-5x faster training than standard transformers
- Automatic memory optimization
- Built-in LoRA + QLoRA support
- Flash Attention 2 integration
- Works seamlessly with our architecture

**Implementation:**
```python
from unsloth import FastLanguageModel

# Load Llama with Unsloth optimizations
predictor_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B',
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,  # QLoRA
)

# Add LoRA adapters (Unsloth-optimized)
predictor_base = FastLanguageModel.get_peft_model(
    predictor_base,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_alpha=32,
    lora_dropout=0.1,
    use_gradient_checkpointing='unsloth',  # optimized checkpointing
)

# Extract layers for predictor
predictor = LlamaPredictor(base_model=predictor_base)
```

**Performance:**
- Training speed: 2-5x faster
- Memory usage: 30-50% reduction
- Quality: Identical to standard approach
- **Highly recommended for this architecture**


### Recommended Configuration

For 8GB VRAM:

```python
# Option 1: QLoRA (best quality/memory balance)
# - 4-bit base models
# - LoRA adapters (r=16)
# - Gradient checkpointing
# - Batch size: 2-4
# Memory: ~4GB

# Option 2: Unsloth + QLoRA (fastest)
# - Unsloth-optimized Llama
# - 4-bit + LoRA
# - Flash Attention 2
# - Batch size: 4-8
# Memory: ~3GB
# Speed: 3-5x faster

# Option 3: Standard LoRA (simpler)
# - fp16 base models
# - LoRA adapters (r=8)
# - Gradient checkpointing
# - Batch size: 1-2
# Memory: ~6GB
```


## Dataset Options

### Video-Text Datasets

#### Large-Scale (VL-JEPA paper used these)

**1. WebVid (2.5M video-text pairs)**
- Source: Stock footage websites
- Duration: 10-30 seconds per video
- Captions: Short descriptions
- Size: ~500GB
- License: Research use
- Best for: General video understanding

```python
from datasets import load_dataset
dataset = load_dataset('webvid', 'webvid-2.5M')
```

**2. CC3M (Conceptual Captions 3M)**
- Source: Web images with alt-text
- Paper used as single-frame videos
- Size: ~30GB
- License: Research use
- Best for: Diverse concepts

**3. HowTo100M (136M clips)**
- Source: Instructional YouTube videos
- ASR-based captions
- Size: Multiple TB
- Best for: Action understanding

#### Medium-Scale (Good for PoC)

**1. MSR-VTT (10k videos)**
- Duration: 10-30 seconds
- 20 captions per video
- Size: ~30GB
- Excellent for initial training

```python
dataset = load_dataset('AlexZigma/msr-vtt')
```

**2. VATEX (41k videos)**
- Multilingual captions (English + Chinese)
- Parallel descriptions
- Size: ~50GB
- Good for diversity

**3. MSVD (120k clips)**
- Microsoft Research Video Description
- Multiple captions per video
- Size: ~15GB

#### Small-Scale (Rapid prototyping)

**1. ActivityNet Captions (20k videos)**
- Temporal annotations
- Dense captions
- Size: ~10GB

**2. COCO + Video augmentation**
- Use COCO images as single frames
- Fast iteration
- Size: ~20GB


### Recommended Approach

**Phase 1: Proof of Concept (This Notebook)**
- Dataset: MSR-VTT (10k videos, manageable size)
- Why: Good quality, reasonable size, multiple captions
- Setup time: ~1 hour download
- Training time: ~6-12 hours on single GPU

**Phase 2: Scale Up**
- Dataset: WebVid subset (100k-500k clips)
- Why: Diverse, high quality, good for generalization
- Training time: 1-3 days

**Phase 3: Full Training (Optional)**
- Datasets: WebVid (full) + CC3M + HowTo100M subset
- Why: Match paper scale
- Training time: 1-2 weeks on multiple GPUs

### Data Loader Implementation

```python
from torch.utils.data import Dataset, DataLoader
import decord  # efficient video loading

class VideoTextDataset(Dataset):
    def __init__(self, video_paths, captions, num_frames=16):
        self.video_paths = video_paths
        self.captions = captions
        self.num_frames = num_frames
        self.tokenizer = y_encoder.tokenizer
    
    def __getitem__(self, idx):
        # Load video
        vr = decord.VideoReader(self.video_paths[idx])
        indices = torch.linspace(0, len(vr)-1, self.num_frames).long()
        frames = vr.get_batch(indices).asnumpy()  # [T, H, W, C]
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        
        # Tokenize caption
        caption = self.captions[idx]
        tokens = self.tokenizer(
            caption,
            return_tensors='pt',
            padding='max_length',
            max_length=77,
            truncation=True,
        )
        
        return {
            'video': frames,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
    
    def __len__(self):
        return len(self.video_paths)

# Usage
train_loader = DataLoader(
    VideoTextDataset(video_paths, captions),
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)
```


## 1. Load Base Components

In [3]:
# X-Encoder: V-JEPA2 (frozen)
# Note: Using placeholder - replace with actual V-JEPA2 model path
# For PoC, we'll simulate the vision encoder output
class MockVJEPA2Encoder(nn.Module):
    """Placeholder for V-JEPA2 ViT-L encoder.
    
    Replace with actual V-JEPA2 model:
    - Output: [batch, seq_len, 1024] embeddings
    - Should be frozen during training
    """
    def __init__(self, hidden_dim: int = 1024):
        super().__init__()
        self.hidden_dim = hidden_dim
        # Freeze this in actual implementation
        self.requires_grad_(False)
    
    def forward(self, video_frames: torch.Tensor) -> torch.Tensor:
        # Mock output: [batch, num_frames, hidden_dim]
        batch_size = video_frames.shape[0]
        num_frames = video_frames.shape[1] if len(video_frames.shape) > 1 else 16
        return torch.randn(batch_size, num_frames, self.hidden_dim, device=video_frames.device)

x_encoder = MockVJEPA2Encoder().to(device)
print(f"X-Encoder loaded (frozen): {sum(p.numel() for p in x_encoder.parameters())} params")

X-Encoder loaded (frozen): 0 params


In [4]:
# Predictor: Last 8 layers of Llama-3.2-1B
# Note: This loads full model then extracts layers - optimize for production

class LlamaPredictor(nn.Module):
    """Predictor using last 8 layers of Llama-3.2-1B.
    
    Non-autoregressive: uses bidirectional attention (no causal mask).
    """
    def __init__(self, model_name: str = "meta-llama/Llama-3.2-1B"):
        super().__init__()
        
        # Load full model to extract layers
        # In production, load only needed layers to save memory
        try:
            full_model = LlamaForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
            
            # Extract last 8 layers (layers 8-15 in 0-indexed 16-layer model)
            self.layers = nn.ModuleList(full_model.model.layers[8:16])
            self.hidden_dim = full_model.config.hidden_size
            
            # Input projection: V-JEPA2 dim (1024) -> Llama dim (2048)
            self.input_proj = nn.Linear(1024, self.hidden_dim)
            
            print(f"Loaded Llama-3.2-1B layers 8-15 ({len(self.layers)} layers)")
            print(f"Hidden dim: {self.hidden_dim}")
            
        except Exception as e:
            print(f"Could not load Llama model: {e}")
            print("Using mock predictor for PoC")
            # Fallback mock
            self.hidden_dim = 2048
            self.input_proj = nn.Linear(1024, self.hidden_dim)
            self.layers = nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=self.hidden_dim,
                    nhead=16,
                    dim_feedforward=self.hidden_dim * 4,
                    batch_first=True,
                )
                for _ in range(8)
            ])
    
    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Forward pass without causal masking (bidirectional).
        
        Args:
            x: [batch, seq_len, 1024] from V-JEPA2
            attention_mask: [batch, seq_len] optional
        
        Returns:
            [batch, seq_len, hidden_dim] predictions
        """
        # Project to Llama dimension
        hidden = self.input_proj(x)
        
        # Pass through layers (no causal mask)
        for layer in self.layers:
            if isinstance(layer, nn.TransformerEncoderLayer):
                # Mock transformer layer
                hidden = layer(hidden)
            else:
                # Real Llama layer
                layer_output = layer(
                    hidden,
                    attention_mask=attention_mask,
                    # Disable causal mask for bidirectional attention
                    use_cache=False,
                )
                hidden = layer_output[0]
        
        return hidden

predictor = LlamaPredictor().to(device)
print(f"Predictor loaded: {sum(p.numel() for p in predictor.parameters())/1e6:.1f}M params")

`torch_dtype` is deprecated! Use `dtype` instead!


Could not load Llama model: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`
Using mock predictor for PoC
Predictor loaded: 405.0M params


In [5]:
# Y-Encoder: EmbeddingGemma-300M
# This encodes text captions into embedding space

class YEncoder(nn.Module):
    """Text encoder using EmbeddingGemma-300M.
    
    Trained with 0.05× learning rate multiplier for stability.
    """
    def __init__(self, model_name: str = "google/embedding-gemma-300m"):
        super().__init__()
        
        try:
            # Load EmbeddingGemma model
            self.model = AutoModel.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.hidden_dim = self.model.config.hidden_size
            
            print(f"Loaded EmbeddingGemma-300M")
            print(f"Hidden dim: {self.hidden_dim}")
            
        except Exception as e:
            print(f"Could not load EmbeddingGemma: {e}")
            print("Using mock Y-Encoder for PoC")
            # Fallback mock
            self.hidden_dim = 2048
            self.model = None
            self.tokenizer = None
    
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Encode text to embeddings.
        
        Args:
            input_ids: [batch, seq_len]
            attention_mask: [batch, seq_len]
        
        Returns:
            [batch, seq_len, hidden_dim] embeddings
        """
        if self.model is None:
            # Mock output
            batch_size, seq_len = input_ids.shape
            return torch.randn(batch_size, seq_len, self.hidden_dim, device=input_ids.device)
        
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        # Return last hidden state
        return outputs.last_hidden_state

y_encoder = YEncoder().to(device)
print(f"Y-Encoder loaded: {sum(p.numel() for p in y_encoder.parameters())/1e6:.1f}M params")

Could not load EmbeddingGemma: google/embedding-gemma-300m is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Using mock Y-Encoder for PoC
Y-Encoder loaded: 0.0M params


## 2. VL-JEPA Architecture Assembly

In [6]:
class VLJEPA(nn.Module):
    """Complete VL-JEPA architecture.
    
    Components:
    - x_encoder: V-JEPA2 (frozen)
    - predictor: Llama layers 8-15 (trainable)
    - y_encoder: EmbeddingGemma (trainable, 0.05× LR)
    - projection heads for alignment
    """
    def __init__(
        self,
        x_encoder: nn.Module,
        predictor: nn.Module,
        y_encoder: nn.Module,
        embed_dim: int = 2048,
    ):
        super().__init__()
        
        self.x_encoder = x_encoder
        self.predictor = predictor
        self.y_encoder = y_encoder
        self.embed_dim = embed_dim
        
        # Projection heads to align predictor and Y-encoder outputs
        self.predictor_proj = nn.Linear(predictor.hidden_dim, embed_dim)
        self.y_encoder_proj = nn.Linear(y_encoder.hidden_dim, embed_dim)
        
        # Temperature for InfoNCE loss
        self.temperature = nn.Parameter(torch.tensor(0.07))
    
    def forward(
        self,
        video_frames: torch.Tensor,
        text_input_ids: torch.Tensor,
        text_attention_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through VL-JEPA.
        
        Args:
            video_frames: [batch, num_frames, ...] video input
            text_input_ids: [batch, seq_len] text tokens
            text_attention_mask: [batch, seq_len] text mask
        
        Returns:
            predicted_embeds: [batch, num_frames, embed_dim]
            target_embeds: [batch, seq_len, embed_dim]
        """
        # X-Encoder: video -> visual embeddings (frozen)
        with torch.no_grad():
            visual_embeds = self.x_encoder(video_frames)
        
        # Predictor: predict text embeddings from video
        predicted_hidden = self.predictor(visual_embeds)
        predicted_embeds = self.predictor_proj(predicted_hidden)
        
        # Y-Encoder: encode actual text
        text_hidden = self.y_encoder(text_input_ids, text_attention_mask)
        target_embeds = self.y_encoder_proj(text_hidden)
        
        # Normalize for contrastive loss
        predicted_embeds = F.normalize(predicted_embeds, dim=-1)
        target_embeds = F.normalize(target_embeds, dim=-1)
        
        return predicted_embeds, target_embeds
    
    def compute_infonce_loss(
        self,
        predicted_embeds: torch.Tensor,
        target_embeds: torch.Tensor,
        text_attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Compute bidirectional InfoNCE loss.
        
        Args:
            predicted_embeds: [batch, num_frames, embed_dim]
            target_embeds: [batch, seq_len, embed_dim]
            text_attention_mask: [batch, seq_len]
        
        Returns:
            Scalar loss
        """
        # Average pooling over non-padding tokens
        # Predicted: average over frames
        pred_pooled = predicted_embeds.mean(dim=1)  # [batch, embed_dim]
        
        # Target: average over non-padding tokens
        mask_expanded = text_attention_mask.unsqueeze(-1).float()  # [batch, seq_len, 1]
        target_masked = target_embeds * mask_expanded
        target_pooled = target_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)  # [batch, embed_dim]
        
        # Normalize
        pred_pooled = F.normalize(pred_pooled, dim=-1)
        target_pooled = F.normalize(target_pooled, dim=-1)
        
        # Compute similarities
        batch_size = pred_pooled.shape[0]
        
        # Video -> Text
        sim_v2t = torch.matmul(pred_pooled, target_pooled.T) / self.temperature  # [batch, batch]
        labels = torch.arange(batch_size, device=sim_v2t.device)
        loss_v2t = F.cross_entropy(sim_v2t, labels)
        
        # Text -> Video (symmetric)
        sim_t2v = sim_v2t.T
        loss_t2v = F.cross_entropy(sim_t2v, labels)
        
        # Bidirectional loss
        return (loss_v2t + loss_t2v) / 2

# Assemble full model
model = VLJEPA(
    x_encoder=x_encoder,
    predictor=predictor,
    y_encoder=y_encoder,
    embed_dim=2048,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal params: {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")
print(f"Frozen params: {(total_params - trainable_params)/1e6:.1f}M")


Total params: 413.4M
Trainable params: 413.4M
Frozen params: 0.0M


## 3. Training Setup

In [7]:
# Optimizer with differential learning rates
# Y-Encoder uses 0.05× multiplier as per paper

base_lr = 1e-4
y_encoder_lr = base_lr * 0.05

# Separate parameter groups
y_encoder_params = list(model.y_encoder.parameters()) + list(model.y_encoder_proj.parameters())
other_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and not any(p is yp for yp in y_encoder_params)
]

optimizer = torch.optim.AdamW([
    {'params': other_params, 'lr': base_lr},
    {'params': y_encoder_params, 'lr': y_encoder_lr},
], weight_decay=0.01)

print(f"Base LR: {base_lr}")
print(f"Y-Encoder LR: {y_encoder_lr}")
print(f"Predictor params: {len(other_params)}")
print(f"Y-Encoder params: {len(y_encoder_params)}")

Base LR: 0.0001
Y-Encoder LR: 5e-06
Predictor params: 101
Y-Encoder params: 2


## 4. Toy Training Loop

This demonstrates the training mechanics with synthetic data.
Replace with real video-caption dataset for actual training.

In [8]:
def create_toy_batch(batch_size: int = 4, num_frames: int = 16, seq_len: int = 32):
    """Create synthetic batch for testing.
    
    Replace with real video-caption dataloader.
    """
    video_frames = torch.randn(batch_size, num_frames, 3, 224, 224).to(device)
    text_input_ids = torch.randint(0, 1000, (batch_size, seq_len)).to(device)
    text_attention_mask = torch.ones(batch_size, seq_len).to(device)
    
    return video_frames, text_input_ids, text_attention_mask

# Training loop
num_steps = 10  # Toy demonstration
model.train()

print("Starting toy training...\n")

for step in range(num_steps):
    # Get batch
    video_frames, text_input_ids, text_attention_mask = create_toy_batch()
    
    # Forward pass
    predicted_embeds, target_embeds = model(
        video_frames,
        text_input_ids,
        text_attention_mask,
    )
    
    # Compute loss
    loss = model.compute_infonce_loss(
        predicted_embeds,
        target_embeds,
        text_attention_mask,
    )
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 2 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f} | Temp: {model.temperature.item():.4f}")

print("\nToy training complete!")

Starting toy training...

Step   0 | Loss: 1.3321 | Temp: 0.0699
Step   2 | Loss: 1.4229 | Temp: 0.0700
Step   4 | Loss: 1.4110 | Temp: 0.0702


KeyboardInterrupt: 

## 5. Validation: Embedding Space Inspection

In [ ]:
# Demonstrate prediction and similarity matching

model.eval()

with torch.no_grad():
    # Get test batch
    video_frames, text_input_ids, text_attention_mask = create_toy_batch(batch_size=2)
    
    # Forward pass
    predicted_embeds, target_embeds = model(
        video_frames,
        text_input_ids,
        text_attention_mask,
    )
    
    # Pool embeddings
    pred_pooled = predicted_embeds.mean(dim=1)  # [2, 2048]
    
    mask_expanded = text_attention_mask.unsqueeze(-1).float()
    target_masked = target_embeds * mask_expanded
    target_pooled = target_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
    
    # Normalize
    pred_pooled = F.normalize(pred_pooled, dim=-1)
    target_pooled = F.normalize(target_pooled, dim=-1)
    
    # Compute similarities
    similarities = torch.matmul(pred_pooled, target_pooled.T)
    
    print("Predicted embeddings shape:", predicted_embeds.shape)
    print("Target embeddings shape:", target_embeds.shape)
    print("\nPooled predicted shape:", pred_pooled.shape)
    print("Pooled target shape:", target_pooled.shape)
    print("\nSimilarity matrix (video -> text):")
    print(similarities.cpu().numpy())
    print("\nDiagonal (correct matches):", similarities.diag().cpu().numpy())
    print("Off-diagonal (incorrect matches):", similarities[~torch.eye(2, dtype=bool)].cpu().numpy())

## 6. Next Steps for Full Implementation

### Data Pipeline
1. Replace mock data with real video-caption dataset
2. Implement frame sampling strategies (dense vs adaptive)
3. Add data augmentation

### Model Components
1. Load actual V-JEPA2 weights (currently mocked)
2. Verify Llama-3.2-1B layer extraction works
3. Test EmbeddingGemma-300M loading

### Training
1. Add gradient checkpointing for memory efficiency
2. Implement learning rate schedule
3. Add validation loop
4. Add checkpointing
5. Scale to multiple GPUs if needed

### Sophia Integration
1. Export trained model for inference
2. Implement Y-Decoder for optional text generation
3. Create HCG node insertion pipeline
4. Add dual embedding (video + text) to all nodes
5. Test cross-modal query (video→text→concept)

### Optimization
1. Profile memory usage (target: 8GB VRAM)
2. Optimize batch size
3. Consider model quantization
4. Implement selective decoding (2.85× reduction from paper)

## Architecture Notes

### Key Differences from Standard Fine-tuning
1. **Joint training**: Predictor and Y-Encoder train together (not post-hoc alignment)
2. **Non-autoregressive**: No causal mask, bidirectional attention
3. **InfoNCE loss**: Contrastive in embedding space, not token prediction
4. **Differential LR**: Y-Encoder uses 0.05× multiplier

### Why This Approach Works Better
- Embedding prediction is more efficient than token prediction
- Joint training ensures predictor learns to output correct embedding space
- Non-autoregressive enables parallel prediction
- Selective decoding maintains text quality while reducing compute

### Memory Budget (8GB Target)
- X-Encoder (frozen): ~304M params = ~608MB (fp16)
- Predictor: ~490M params = ~980MB (fp16)
- Y-Encoder: ~300M params = ~600MB (fp16)
- Activations + gradients: ~5GB
- Batch size 1, gradient checkpointing recommended